# Position Encoding - Sinusodial, ROPE, ALiBi

Attention is permutation-invariant. "The cat sat on the mat" and "mat the on sat cat the" produce the same output without positional signal.

## Problem definition

Scaled dot-product attention is order-blind. The attention matrix is computed from pairwise similarities, Shuffle the rows of X, get the rows of the output shuffled the same sway. 

That is not a bug in a bag-of-words model. For language, code, audio, video -- anything where order carries meaning -- it is faital.

### Three eras of answers

### Absolute sinusoidal

**Add `sin/cos` of position** to the embedding. Simple, learnable-free, extrapolates poorly beyound trained lengths.

### RoPE -- Rotary Position Embedding

**Rotate Q and K vectors** by an angle proportinal to position. Encdes relative position directly in the dot product. 

### ALiBi -- Attention with Linear Biases

Skip embedding entirely. Add a per-head linear penalty to **attention score** based on distance. Excellent length extrapolation.

## Basic Concept

### Absolute sinusoidal

Pre-compute a fixed matrix `PE` of shape `(max_len, d_model)`:

```
PE[pos, 2i] = sin(pos / 10000^(2i / d_model))
PE[pos, 2i + 1] = cos(pos / 10000^(2i / d_model))
```

Then `X' = X + PE[:N]` before attention. Each dimension is a sinusoid at a difference frequency.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter

import numpy as np
import matplotlib.pyplot as plt


def sinusoidal_pe(max_len: int, d_model: int) -> np.ndarray:
    pe = np.zeros((max_len, d_model))
    position = np.arange(max_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * (-np.log(10_000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe


import sys
from pathlib import Path

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter

import numpy as np
import matplotlib.pyplot as plt


def sinusoidal_pe(max_len: int, d_model: int) -> np.ndarray:
    pe = np.zeros((max_len, d_model))
    position = np.arange(max_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * (-np.log(10_000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe


def pair_wavelength(pair_idx: int, d_model: int) -> float:
    """One full sin/cos cycle spans this many token positions."""
    return 2 * np.pi * (10_000 ** (2 * pair_idx / d_model))


with SectionPrinter("Visualizing Absolute Sinusoidal"):
    max_len = 128
    d_model = 128  # smaller dim so wavelengths are visible on screen
    pe = sinusoidal_pe(max_len, d_model)

    # three frequency bands: slow / medium / fast
    pairs = [0, 8, 32]
    labels = ["low freq — slow rotation", "mid freq", "high freq — fast rotation"]
    n_steps = 60

    fig = plt.figure(figsize=(14, 9))
    gs = fig.add_gridspec(3, 3, height_ratios=[1.1, 0.9, 1.0])

    # --- row 1: each (sin, cos) pair traces a unit circle as position advances ---
    for col, (pair_i, label) in enumerate(zip(pairs, labels)):
        ax = fig.add_subplot(gs[0, col])
        sin_d = pe[:n_steps, 2 * pair_i]
        cos_d = pe[:n_steps, 2 * pair_i + 1]
        colors = plt.cm.viridis(np.linspace(0, 1, n_steps))
        for j in range(n_steps - 1):
            ax.plot(sin_d[j : j + 2], cos_d[j : j + 2], color=colors[j], lw=2)
        ax.scatter(sin_d[0], cos_d[0], c="#2ecc71", s=50, zorder=5, label="pos=0")
        ax.scatter(sin_d[-1], cos_d[-1], c="#e74c3c", s=50, zorder=5, label=f"pos={n_steps - 1}")
        circle = plt.Circle((0, 0), 1, fill=False, color="gray", ls="--", lw=0.8)
        ax.add_patch(circle)
        ax.set_xlim(-1.15, 1.15)
        ax.set_ylim(-1.15, 1.15)
        ax.set_aspect("equal")
        ax.set_xlabel("sin")
        ax.set_ylabel("cos")
        wl = pair_wavelength(pair_i, d_model)
        ax.set_title(f"{label}\npair {pair_i}, λ ≈ {wl:.0f} tokens")
        ax.legend(fontsize=7, loc="upper right")

    # --- row 2: sin wave over position, vertical ticks = one wavelength ---
    x = np.arange(80)
    for col, pair_i in enumerate(pairs):
        ax = fig.add_subplot(gs[1, col])
        ax.plot(x, pe[:80, 2 * pair_i], lw=2)
        wl = pair_wavelength(pair_i, d_model)
        for m in np.arange(0, 80, wl):
            ax.axvline(m, color="orange", alpha=0.45, lw=1)
        ax.set_ylim(-1.15, 1.15)
        ax.set_xlabel("position")
        ax.set_ylabel("sin value")
        ax.set_title(f"orange lines = one wavelength ({wl:.0f} tokens)")
        ax.grid(True, alpha=0.3)

    # --- row 3: dot-product similarity — nearby positions are similar ---
    ax = fig.add_subplot(gs[2, :])
    n_pos = 40
    pe_slice = pe[:n_pos]
    sim = pe_slice @ pe_slice.T
    im = ax.imshow(sim, cmap="magma", aspect="auto", origin="lower")
    ax.set_xlabel("position j")
    ax.set_ylabel("position i")
    ax.set_title(
        "PE(i) · PE(j): bright diagonal = nearby tokens have similar encodings; "
        "pattern encodes relative distance"
    )
    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)

    fig.suptitle(
        "Absolute sinusoidal PE: each dim-pair spins on a circle; "
        "many frequencies combined = unique fingerprint per position",
        y=1.01,
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()


### RoPE — Rotary Position Embedding

Instead of **adding** position vectors, RoPE **rotates** each consecutive dimension pair `(2i, 2i+1)` in Q and K:

```
θ_i = 10000^(-2i / d_model)        # rotation per token (radians)
R_i(m) = [[cos(m·θ_i), -sin(m·θ_i)],
          [sin(m·θ_i),  cos(m·θ_i)]]

[q'_{2i}, q'_{2i+1}]ᵀ = R_i(m) · [q_{2i}, q_{2i+1}]ᵀ
```

Key property: `⟨RoPE(q,m), RoPE(k,n)⟩` depends only on **m − n** (relative distance), not absolute positions.

In [ ]:
def rope_thetas(d_model: int, base: float = 10_000.0) -> np.ndarray:
    """Per-pair rotation angle (radians) for each +1 token position."""
    pair_idx = np.arange(d_model // 2)
    return base ** (-2 * pair_idx / d_model)


with SectionPrinter("Visualizing RoPE"):
    d_model = 128
    thetas = rope_thetas(d_model)
    pair_indices = np.arange(d_model // 2)
    wavelengths = 2 * np.pi / thetas

    fig = plt.figure(figsize=(14, 11))
    gs = fig.add_gridspec(3, 3, height_ratios=[0.9, 1.1, 1.0])

    # --- row 1: rotation unit θ_i from low → high pair index ---
    ax_theta = fig.add_subplot(gs[0, :2])
    ax_theta.plot(pair_indices, thetas, "o-", lw=2, ms=4)
    ax_theta.set_yscale("log")
    ax_theta.set_xlabel("pair index i  (low → high)")
    ax_theta.set_ylabel("θ_i  (radians per token, log scale)")
    ax_theta.set_title("RoPE: rotation per +1 position — high i spins slower")
    ax_theta.grid(True, alpha=0.3)
    for mark_i in [0, 8, 32, 63]:
        ax_theta.annotate(
            f"i={mark_i}\nθ={thetas[mark_i]:.2e}",
            (mark_i, thetas[mark_i]),
            textcoords="offset points",
            xytext=(6, 6),
            fontsize=8,
        )

    ax_wl = fig.add_subplot(gs[0, 2])
    ax_wl.plot(pair_indices, wavelengths, color="C1", lw=2)
    ax_wl.set_yscale("log")
    ax_wl.set_xlabel("pair index i")
    ax_wl.set_ylabel("wavelength λ (tokens)")
    ax_wl.set_title("one full turn needs λ tokens")
    ax_wl.grid(True, alpha=0.3)

    # --- row 2: circle trajectories (start from [1,0], rotate by pos·θ_i) ---
    pairs = [0, 8, 32]
    labels = ["low i — fast spin", "mid i", "high i — slow spin"]
    n_steps = 60
    for col, (pair_i, label) in enumerate(zip(pairs, labels)):
        ax = fig.add_subplot(gs[1, col])
        theta = thetas[pair_i]
        sin_d = np.sin(np.arange(n_steps) * theta)
        cos_d = np.cos(np.arange(n_steps) * theta)
        colors = plt.cm.plasma(np.linspace(0, 1, n_steps))
        for j in range(n_steps - 1):
            ax.plot(sin_d[j : j + 2], cos_d[j : j + 2], color=colors[j], lw=2)
        ax.scatter(sin_d[0], cos_d[0], c="#2ecc71", s=50, zorder=5, label="pos=0")
        ax.scatter(sin_d[-1], cos_d[-1], c="#e74c3c", s=50, zorder=5, label=f"pos={n_steps - 1}")
        circle = plt.Circle((0, 0), 1, fill=False, color="gray", ls="--", lw=0.8)
        ax.add_patch(circle)
        ax.set_xlim(-1.15, 1.15)
        ax.set_ylim(-1.15, 1.15)
        ax.set_aspect("equal")
        ax.set_xlabel("dim 2i")
        ax.set_ylabel("dim 2i+1")
        wl = wavelengths[pair_i]
        ax.set_title(f"{label}\nθ={theta:.2e} rad/token, λ≈{wl:.0f}")
        ax.legend(fontsize=7)

    # --- row 3: RoPE dot product depends only on m−n ---
    ax_rope = fig.add_subplot(gs[2, 0])
    n_pos = 30
    m_grid, n_grid = np.meshgrid(np.arange(n_pos), np.arange(n_pos), indexing="ij")
    rel = m_grid - n_grid
    rope_sim = np.cos(rel * thetas[0])  # single pair, q=k=[1,0]
    im1 = ax_rope.imshow(rope_sim, cmap="RdBu", vmin=-1, vmax=1, origin="lower")
    ax_rope.set_xlabel("position n")
    ax_rope.set_ylabel("position m")
    ax_rope.set_title("RoPE pair 0: cos((m−n)·θ₀)\nconstant along diagonals")
    plt.colorbar(im1, ax=ax_rope, fraction=0.046)

    ax_abs = fig.add_subplot(gs[2, 1:])
    pe_full = sinusoidal_pe(n_pos, d_model)
    abs_sim = pe_full @ pe_full.T
    im2 = ax_abs.imshow(abs_sim, cmap="magma", aspect="auto", origin="lower")
    ax_abs.set_xlabel("position n")
    ax_abs.set_ylabel("position m")
    ax_abs.set_title(
        "Absolute PE: PE(m)·PE(n) — pattern is not purely a function of m−n"
    )
    plt.colorbar(im2, ax=ax_abs, fraction=0.02, pad=0.02)

    fig.suptitle(
        "RoPE: each pair has its own θ_i; rotation encodes position; attention sees m−n",
        y=1.01,
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()